In [11]:
"""
BMNB (Blended Multi-group Naive Bayes) - Fairness-Aware Classifier
====================================================================
Corrected pipeline addressing:
  1. Data leakage: LabelEncoder / dropna no longer fit on the full
     dataset before the train/val/test split. Encoders and scalers are
     fit on the training fold only and applied (transform-only) to
     val/test, with explicit handling of unseen categories.
  2. Sensitive-attribute recovery for SMOTEENN-synthesized rows now
     runs the nearest-neighbor search on SCALED features, so the
     recovery isn't dominated by features with large numeric ranges.
  3. BMNB.predict_proba no longer silently zeroes out rows whose
     sensitive-attribute value was not seen during training; it falls
     back to the global model's probability, consistent with
     explain_decision().
  4. Removed redundant re-fitting of a second GaussianNB in
     "threshold" mode (reuses model.global_model) and removed the
     duplicate explain_decision() call.
  5. class_priors default to being estimated from the (resampled)
     training data rather than a silently hard-coded [0.4, 0.6],
     while still allowing an explicit override.
  6. repeated_experiment is explicitly documented as repeated
     hold-out validation (not k-fold); a genuine StratifiedKFold
     variant is provided separately (see `run_dataset_kfold`) for
     reviewers who specifically asked for k-fold CV.
  7. Unused MinMaxScaler import is now actually used; fairness-metric
     group ordering/direction is documented explicitly.
"""

import numpy as np
import pandas as pd

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.neighbors import NearestNeighbors

from imblearn.combine import SMOTEENN

from scipy.stats import wilcoxon

import matplotlib.pyplot as plt
import seaborn as sns


# ==========================================================
# Dataset-specific raw preprocessing
# (row filtering / label extraction ONLY - no encoder fitting
#  here, since that must happen strictly after the train/test
#  split to avoid leakage)
# ==========================================================

def preprocess_data(df, dataset_name):
    """
    Apply dataset-specific row filtering and extract X (raw feature
    DataFrame, may still contain object/categorical columns), y
    (target), and s (sensitive attribute).

    NOTE: No encoding or scaling is done here. That is intentional -
    fitting a LabelEncoder/Scaler here (before train/test split) would
    leak test-set information into preprocessing. Encoding/scaling is
    performed later, inside evaluate_bmnb, using train-fold-only fits.

    Arguments:
        df: Raw input DataFrame
        dataset_name: One of ['adult', 'propublica', 'framingham']

    Returns:
        X_df: pd.DataFrame of raw (unencoded) features
        y: np.ndarray target
        s: np.ndarray sensitive attribute
        cat_cols: list of column names that are categorical/object
                  and need encoding downstream
    """
    df = df.copy()

    if dataset_name == 'adult':
        df.replace("?", np.nan, inplace=True)
        df.dropna(inplace=True)
        df.reset_index(drop=True, inplace=True)

        y = df['income'].apply(lambda x: 1 if '>50K' in x else 0).values
        s = df['sex'].apply(lambda x: 1 if x.strip() == 'Male' else 0).values

        X_df = df.drop(['income', 'sex'], axis=1)

    elif dataset_name == 'propublica':
        df.dropna(inplace=True)
        df.reset_index(drop=True, inplace=True)

        y = df['Two_yr_Recidivism'].values
        s = df['African_American'].values

        race_columns = ['African_American', 'Asian', 'Hispanic',
                         'Native_American', 'Other']
        X_df = df.drop(['Two_yr_Recidivism'] + race_columns, axis=1)

    elif dataset_name == 'framingham':
        df.dropna(inplace=True)
        df.reset_index(drop=True, inplace=True)

        y = df['TenYearCHD'].values
        s = df['male'].values

        X_df = df.drop(['TenYearCHD', 'male'], axis=1)

    else:
        raise ValueError(
            "Unsupported dataset_name. Use 'adult', 'propublica', or 'framingham'."
        )

    cat_cols = [
        col for col in X_df.columns
        if X_df[col].dtype == 'object' or isinstance(X_df[col].iloc[0], str)
    ]

    return X_df, y, s, cat_cols


# ==========================================================
# Train-fold-only encoding / scaling
# ==========================================================

def fit_encoders(X_train_df, cat_cols):
    """
    Fit one LabelEncoder per categorical column, using TRAINING data
    only. Returns a dict {col: fitted LabelEncoder}.
    """
    encoders = {}

    for col in cat_cols:
        le = LabelEncoder()
        le.fit(X_train_df[col].astype(str))
        encoders[col] = le

    return encoders


def apply_encoders(X_df, cat_cols, encoders):
    """
    Transform categorical columns using encoders fit on the training
    fold. Categories unseen during training are mapped to -1 rather
    than raising, since val/test can legitimately contain unseen
    categories.
    """
    X_df = X_df.copy()

    for col in cat_cols:
        le = encoders[col]
        known = set(le.classes_)

        X_df[col] = X_df[col].astype(str).apply(
            lambda v: le.transform([v])[0] if v in known else -1
        )

    return X_df


def encode_and_scale(X_train_df, X_val_df, X_test_df, cat_cols):
    """
    Fit LabelEncoders and a MinMaxScaler on X_train only; apply
    (transform-only) to train/val/test. Returns numpy arrays plus the
    fitted encoders/scaler in case they're needed later (e.g. for
    inference on new data).
    """
    encoders = fit_encoders(X_train_df, cat_cols)

    X_train_enc = apply_encoders(X_train_df, cat_cols, encoders)
    X_val_enc = apply_encoders(X_val_df, cat_cols, encoders)
    X_test_enc = apply_encoders(X_test_df, cat_cols, encoders)

    scaler = MinMaxScaler()
    X_train_arr = scaler.fit_transform(X_train_enc.values.astype(float))
    X_val_arr = scaler.transform(X_val_enc.values.astype(float))
    X_test_arr = scaler.transform(X_test_enc.values.astype(float))

    return X_train_arr, X_val_arr, X_test_arr, encoders, scaler


# ==========================================================
# BMNB Class
# ==========================================================

class BMNB:

    def __init__(self, alpha=0.5, class_priors=None):
        self.alpha = alpha
        self.class_priors = class_priors  # may be None -> estimated at fit time

        self.global_model = None
        self.group_models = {}
        self.groups = None
        self.thresholds = {}

    # ======================================================
    # Train
    # ======================================================

    def fit(self, X, y, sensitive):

        priors = self._resolve_priors(y)

        self.global_model = GaussianNB(priors=priors)
        self.global_model.fit(X, y)

        self.groups = np.unique(sensitive)
        self.group_models = {}

        for group in self.groups:
            idx = (sensitive == group)

            # A group-specific model needs both classes present;
            # fall back to sharing the global model if a group is
            # degenerate (e.g. too few samples / single class after
            # a particular resampling draw).
            if len(np.unique(y[idx])) < 2:
                self.group_models[group] = self.global_model
                continue

            group_priors = self._resolve_priors(y[idx])
            model = GaussianNB(priors=group_priors)
            model.fit(X[idx], y[idx])
            self.group_models[group] = model

        return self

    def _resolve_priors(self, y):
        """
        Use explicit class_priors if provided at construction time;
        otherwise estimate priors from the observed class frequencies
        of `y`, so the model doesn't silently assume a fixed
        imbalance ratio unless the user asked for one.
        """
        if self.class_priors is not None:
            return self.class_priors

        classes, counts = np.unique(y, return_counts=True)
        if len(classes) < 2:
            return None  # let GaussianNB fall back to its own default

        freqs = counts / counts.sum()
        # order priors to match class labels [0, 1]
        order = np.argsort(classes)
        return list(freqs[order])

    # ======================================================
    # Blended Likelihood Estimation
    # ======================================================

    def predict_proba(self, X, sensitive):

        global_probs = self.global_model.predict_proba(X)[:, 1]

        # Fallback to the global model's probability for any row
        # whose sensitive-group value wasn't seen at training time,
        # instead of silently leaving it at 0.0 (predicting negative).
        blended_probs = global_probs.copy()

        for group in self.groups:
            idx = (sensitive == group)

            if np.sum(idx) == 0:
                continue

            group_probs = self.group_models[group].predict_proba(X[idx])[:, 1]

            blended_probs[idx] = (
                self.alpha * group_probs
                + (1 - self.alpha) * global_probs[idx]
            )

        return blended_probs

    # ======================================================
    # Adaptive Threshold Calibration
    # Fairness-Optimized Threshold Search
    # ======================================================

    def calibrate_thresholds(self, y_true, probabilities, sensitive):

        self.thresholds = {}

        candidate_thresholds = np.linspace(0.05, 0.95, 91)
        groups = np.unique(sensitive)

        current_thresholds = {g: 0.50 for g in groups}

        for target_group in groups:

            best_threshold = 0.50
            best_fs = -np.inf

            for threshold in candidate_thresholds:

                temp_thresholds = current_thresholds.copy()
                temp_thresholds[target_group] = threshold

                y_pred = np.zeros(len(probabilities), dtype=int)

                for g in groups:
                    idx = (sensitive == g)
                    y_pred[idx] = (probabilities[idx] >= temp_thresholds[g]).astype(int)

                group_metrics = calculate_group_metrics(y_true, y_pred, sensitive)
                fairness = calculate_fairness_metrics(group_metrics)
                fs = fairness["FS"]

                if fs > best_fs:
                    best_fs = fs
                    best_threshold = threshold

            current_thresholds[target_group] = best_threshold

        self.thresholds = current_thresholds
        return self.thresholds

    def predict(self, X, sensitive):

        probs = self.predict_proba(X, sensitive)
        predictions = np.zeros(len(probs), dtype=int)

        for group in np.unique(sensitive):
            idx = (sensitive == group)
            threshold = self.thresholds.get(group, 0.5)
            predictions[idx] = (probs[idx] >= threshold).astype(int)

        return predictions

    # ======================================================
    # Decision Explanation
    # ======================================================

    def explain_decision(self, X, sensitive, return_dataframe=True):

        global_probs = self.global_model.predict_proba(X)[:, 1]
        blended_probs = self.predict_proba(X, sensitive)

        rows = []

        for i in range(len(X)):

            group = sensitive[i]
            threshold = self.thresholds.get(group, 0.5)

            if group in self.group_models:
                group_prob = self.group_models[group].predict_proba(X[i:i + 1])[:, 1][0]
            else:
                group_prob = global_probs[i]

            blended = blended_probs[i]
            prediction = int(blended >= threshold)
            margin = blended - threshold

            explanation = (
                f"Group={group}; "
                f"Global={global_probs[i]:.3f}; "
                f"Group={group_prob:.3f}; "
                f"Alpha={self.alpha:.2f}; "
                f"Blended={blended:.3f}; "
                f"Threshold={threshold:.3f}; "
                f"Prediction={prediction}; "
                f"Margin={margin:.3f}"
            )

            rows.append({
                "index": i,
                "group": group,
                "global_probability": round(global_probs[i], 4),
                "group_probability": round(group_prob, 4),
                "alpha": self.alpha,
                "blended_probability": round(blended, 4),
                "threshold": round(threshold, 4),
                "margin_to_threshold": round(margin, 4),
                "prediction": prediction,
                "decision": "Positive" if prediction else "Negative",
                "explanation": explanation
            })

        if return_dataframe:
            return pd.DataFrame(rows)

        return rows


# ==========================================================
# Group Metrics
# ==========================================================

def calculate_group_metrics(y_true, y_pred, sensitive):

    metrics = {}

    for group in np.unique(sensitive):
        idx = (sensitive == group)

        yt = y_true[idx]
        yp = y_pred[idx]

        TP = np.sum((yt == 1) & (yp == 1))
        TN = np.sum((yt == 0) & (yp == 0))
        FP = np.sum((yt == 0) & (yp == 1))
        FN = np.sum((yt == 1) & (yp == 0))

        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
        PPR = np.mean(yp)

        metrics[group] = {
            "TPR": TPR,
            "FPR": FPR,
            "PPR": PPR,
            "Support": len(yt)
        }

    return metrics


# ==========================================================
# Fairness Metrics
#
# NOTE on direction: groups are sorted ascending and g1 (the larger
# sorted value) is treated as the "reference" group whose rates are
# subtracted from / divided into g0's. For all three datasets here,
# sensitive attributes are coded so that 1 = the historically
# advantaged/majority group (Male for 'adult'/'framingham';
# African_American=1 is the group of fairness interest for
# 'propublica' -- confirm this matches your intended interpretation
# direction in the manuscript, since it flips the sign of EOD/SPD).
# ==========================================================

def calculate_fairness_metrics(group_metrics):

    groups = sorted(group_metrics.keys())

    if len(groups) != 2:
        raise ValueError(
            "Fairness metrics currently support binary sensitive groups only"
        )

    g0, g1 = groups[0], groups[1]

    EOD = group_metrics[g1]["TPR"] - group_metrics[g0]["TPR"]
    EMOD = group_metrics[g1]["FPR"] - group_metrics[g0]["FPR"]
    SPD = group_metrics[g1]["PPR"] - group_metrics[g0]["PPR"]

    DI = (
        group_metrics[g1]["PPR"] / group_metrics[g0]["PPR"]
        if group_metrics[g0]["PPR"] > 0 else 0
    )

    BI = (abs(EOD) + abs(EMOD) + abs(SPD) + abs(1 - DI)) / 4
    FS = 1 - BI

    return {
        "EOD": EOD,
        "EMOD": EMOD,
        "SPD": SPD,
        "DI": DI,
        "BI": BI,
        "FS": FS
    }


# ==========================================================
# Dataset Labels
# ==========================================================

def get_label_mapping(dataset_name):

    mappings = {
        "adult": {0: "<=50K", 1: ">50K"},
        "propublica": {0: "No Recidivism", 1: "Recidivism"},
        "framingham": {0: "No Stroke", 1: "Stroke"}
    }

    if dataset_name not in mappings:
        raise ValueError("Unknown dataset.")

    return mappings[dataset_name]


# ==========================================================
# Confusion Matrix
# ==========================================================

def plot_confusion_matrix(y_true, y_pred, dataset_name):

    labels = get_label_mapping(dataset_name)
    unique = np.unique(np.concatenate((y_true, y_pred)))
    display = [labels[i] for i in unique]

    cm = confusion_matrix(y_true, y_pred, labels=unique)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
                xticklabels=display, yticklabels=display)
    plt.title(f"Confusion Matrix ({dataset_name})")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


# ==========================================================
# Core Evaluation (leakage-free)
# ==========================================================

def evaluate_bmnb(X_df, y, s, dataset_name, cat_cols,
                   alpha=0.5, class_priors=None,
                   random_state=42, mode="full", verbose=True):
    """
    X_df: raw (unencoded) feature DataFrame, as returned by preprocess_data
    y, s: target and sensitive-attribute arrays
    cat_cols: categorical column names in X_df, from preprocess_data
    class_priors: optional explicit [p0, p1]; if None, priors are
                  estimated from the resampled training labels.
    """

    # ------------------------------------------------------------
    # 1. Split FIRST (raw, unencoded data) - test set stays untouched
    # ------------------------------------------------------------
    X_train_full_df, X_test_df, y_train_full, y_test, s_train_full, s_test = train_test_split(
        X_df, y, s,
        test_size=0.20,
        stratify=y,
        random_state=random_state
    )

    X_train_df, X_val_df, y_train, y_val, s_train, s_val = train_test_split(
        X_train_full_df, y_train_full, s_train_full,
        test_size=0.20,
        stratify=y_train_full,
        random_state=random_state
    )

    # ------------------------------------------------------------
    # 2. Encode + scale using TRAIN-FOLD-ONLY fits
    # ------------------------------------------------------------
    X_train, X_val, X_test, encoders, scaler = encode_and_scale(
        X_train_df, X_val_df, X_test_df, cat_cols
    )

    # ------------------------------------------------------------
    # 3. Balance the training set only
    # ------------------------------------------------------------
    sm = SMOTEENN(random_state=random_state)
    X_train_resampled, y_train_resampled = sm.fit_resample(X_train, y_train)

    # ------------------------------------------------------------
    # 4. Recover sensitive attribute for resampled rows.
    #    NN search now runs on SCALED features (both fit and query
    #    side), so it isn't dominated by large-range features.
    # ------------------------------------------------------------
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(X_train)
    _, indices = nn.kneighbors(X_train_resampled)
    s_train_resampled = s_train[indices.flatten()]

    # ------------------------------------------------------------
    # 5. Train BMNB (priors estimated from resampled train labels
    #    unless explicitly overridden)
    # ------------------------------------------------------------
    model = BMNB(alpha=alpha, class_priors=class_priors)
    model.fit(X_train_resampled, y_train_resampled, s_train_resampled)

    # ------------------------------------------------------------
    # 6. Prediction, by mode
    # ------------------------------------------------------------
    if mode.lower() == "threshold":
        # Global Naive Bayes (reuse model.global_model - no need to
        # refit a second, identical GaussianNB) + adaptive threshold
        val_probs = model.global_model.predict_proba(X_val)[:, 1]

        model.calibrate_thresholds(y_val, val_probs, s_val)

        test_probs = model.global_model.predict_proba(X_test)[:, 1]

        y_pred = np.zeros(len(test_probs), dtype=int)
        for group in model.groups:
            idx = (s_test == group)
            threshold = model.thresholds.get(group, 0.5)
            y_pred[idx] = (test_probs[idx] >= threshold).astype(int)

    elif mode.lower() == "ble":
        # Blended Likelihood + fixed 0.5 threshold
        test_probs = model.predict_proba(X_test, s_test)
        y_pred = (test_probs >= 0.5).astype(int)

    elif mode.lower() == "full":
        # Blended Likelihood + adaptive threshold
        val_probs = model.predict_proba(X_val, s_val)
        model.calibrate_thresholds(y_val, val_probs, s_val)
        y_pred = model.predict(X_test, s_test)

    else:
        raise ValueError("mode must be 'threshold', 'ble', or 'full'.")

    # ------------------------------------------------------------
    # 7. Decision Explanation (computed once, reused for both the
    #    verbose printout and the return value if needed)
    # ------------------------------------------------------------
    explanations = model.explain_decision(X_test, s_test, return_dataframe=True)

    if verbose:
        print("\nDecision Explanation Samples")
        print(explanations[[
            "index", "group", "global_probability", "group_probability",
            "blended_probability", "threshold", "margin_to_threshold", "decision"
        ]].head(10))

        print("\nExample Explanation")
        print(explanations.loc[0, "explanation"])

    # ------------------------------------------------------------
    # 8. Evaluation metrics
    # ------------------------------------------------------------
    accuracy = accuracy_score(y_test, y_pred)
    group_metrics = calculate_group_metrics(y_test, y_pred, s_test)
    fairness = calculate_fairness_metrics(group_metrics)

    if verbose:
        print("Accuracy:", accuracy)
        print("BI:", fairness["BI"])
        print("FS:", fairness["FS"])

    return {
        "accuracy": accuracy,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "BI": fairness["BI"],
        "FS": fairness["FS"],
        "EOD": fairness["EOD"],
        "EMOD": fairness["EMOD"],
        "SPD": fairness["SPD"],
        "DI": fairness["DI"],
        "explanations": explanations
    }


# ==========================================================
# Repeated Experiment (REPEATED HOLD-OUT, not k-fold)
# Each repetition redraws a fresh 80/20 -> 80/20 split with a new
# random_state and reapplies SMOTEENN independently. Use this to
# estimate variance across resampling/splitting randomness. If your
# reviewers specifically want k-fold CV, use run_dataset_kfold below.
# ==========================================================

def repeated_experiment(X_df, y, s, dataset_name, cat_cols,
                         repeats=10, alpha=0.5, class_priors=None):

    methods = {
        "Threshold Only": "threshold",
        "BLE Only": "ble",
        "Full BMNB": "full"
    }

    results = {}

    for method, mode in methods.items():
        print("\nRunning:", method)
        results[method] = []

        for seed in range(repeats):
            print(f"  Repetition {seed + 1}/{repeats}")

            metrics = evaluate_bmnb(
                X_df, y, s, dataset_name, cat_cols,
                alpha=alpha, class_priors=class_priors,
                random_state=seed, mode=mode, verbose=False
            )

            results[method].append(metrics)

    return results


# ==========================================================
# Genuine K-Fold Cross-Validation variant
# (added to directly satisfy a reviewer request for k-fold CV,
#  distinct from the repeated hold-out procedure above)
# ==========================================================

def evaluate_bmnb_kfold(X_df, y, s, dataset_name, cat_cols,
                         alpha=0.5, class_priors=None,
                         n_splits=5, random_state=42, mode="full", verbose=False):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    X_arr = X_df.reset_index(drop=True)
    y_arr = np.asarray(y)
    s_arr = np.asarray(s)

    fold_results = []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_arr, y_arr)):

        X_train_full_df = X_arr.iloc[train_idx].reset_index(drop=True)
        X_test_df = X_arr.iloc[test_idx].reset_index(drop=True)
        y_train_full, y_test = y_arr[train_idx], y_arr[test_idx]
        s_train_full, s_test = s_arr[train_idx], s_arr[test_idx]

        # carve out a validation split from the training fold only,
        # for threshold calibration (test fold stays fully untouched)
        X_train_df, X_val_df, y_train, y_val, s_train, s_val = train_test_split(
            X_train_full_df, y_train_full, s_train_full,
            test_size=0.20, stratify=y_train_full, random_state=random_state
        )

        X_train, X_val, X_test, encoders, scaler = encode_and_scale(
            X_train_df, X_val_df, X_test_df, cat_cols
        )

        sm = SMOTEENN(random_state=random_state)
        X_train_resampled, y_train_resampled = sm.fit_resample(X_train, y_train)

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_train)
        _, indices = nn.kneighbors(X_train_resampled)
        s_train_resampled = s_train[indices.flatten()]

        model = BMNB(alpha=alpha, class_priors=class_priors)
        model.fit(X_train_resampled, y_train_resampled, s_train_resampled)

        if mode.lower() == "threshold":
            val_probs = model.global_model.predict_proba(X_val)[:, 1]
            model.calibrate_thresholds(y_val, val_probs, s_val)
            test_probs = model.global_model.predict_proba(X_test)[:, 1]
            y_pred = np.zeros(len(test_probs), dtype=int)
            for group in model.groups:
                idx = (s_test == group)
                threshold = model.thresholds.get(group, 0.5)
                y_pred[idx] = (test_probs[idx] >= threshold).astype(int)

        elif mode.lower() == "ble":
            test_probs = model.predict_proba(X_test, s_test)
            y_pred = (test_probs >= 0.5).astype(int)

        elif mode.lower() == "full":
            val_probs = model.predict_proba(X_val, s_val)
            model.calibrate_thresholds(y_val, val_probs, s_val)
            y_pred = model.predict(X_test, s_test)

        else:
            raise ValueError("mode must be 'threshold', 'ble', or 'full'.")

        accuracy = accuracy_score(y_test, y_pred)
        group_metrics = calculate_group_metrics(y_test, y_pred, s_test)
        fairness = calculate_fairness_metrics(group_metrics)

        if verbose:
            print(f"Fold {fold_idx + 1}/{n_splits} - Accuracy: {accuracy:.3f}, FS: {fairness['FS']:.3f}")

        fold_results.append({
            "accuracy": accuracy,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "BI": fairness["BI"],
            "FS": fairness["FS"],
            "EOD": fairness["EOD"],
            "EMOD": fairness["EMOD"],
            "SPD": fairness["SPD"],
            "DI": fairness["DI"],
        })

    return fold_results


# ==========================================================
# Result Summary
# ==========================================================

def summarize_results(results):

    summary = []

    for method, values in results.items():

        accuracy = [x["accuracy"] for x in values]
        precision = [x["precision"] for x in values]
        recall = [x["recall"] for x in values]
        f1 = [x["f1"] for x in values]

        BI = [x["BI"] for x in values]
        FS = [x["FS"] for x in values]
        EOD = [x["EOD"] for x in values]
        EMOD = [x["EMOD"] for x in values]
        SPD = [x["SPD"] for x in values]
        DI = [x["DI"] for x in values]

        summary.append({
            "Method": method,
            "Accuracy": f"{np.mean(accuracy):.3f} \u00b1 {np.std(accuracy):.3f}",
            "Precision": f"{np.mean(precision):.3f} \u00b1 {np.std(precision):.3f}",
            "Recall": f"{np.mean(recall):.3f} \u00b1 {np.std(recall):.3f}",
            "F1": f"{np.mean(f1):.3f} \u00b1 {np.std(f1):.3f}",
            "BI": f"{np.mean(BI):.3f} \u00b1 {np.std(BI):.3f}",
            "FS": f"{np.mean(FS):.3f} \u00b1 {np.std(FS):.3f}",
            "EOD": f"{np.mean(EOD):.3f} \u00b1 {np.std(EOD):.3f}",
            "EMOD": f"{np.mean(EMOD):.3f} \u00b1 {np.std(EMOD):.3f}",
            "SPD": f"{np.mean(SPD):.3f} \u00b1 {np.std(SPD):.3f}",
            "DI": f"{np.mean(DI):.3f} \u00b1 {np.std(DI):.3f}",
        })

    return pd.DataFrame(summary)


# ==========================================================
# Wilcoxon Signed Rank Test
# ==========================================================

def wilcoxon_test(results, method1, method2, metric):

    values1 = [x[metric] for x in results[method1]]
    values2 = [x[metric] for x in results[method2]]

    statistic, p_value = wilcoxon(values1, values2)

    return {
        "Comparison": f"{method1} vs {method2}",
        "Metric": metric,
        "Statistic": round(statistic, 3),
        "p-value": round(p_value, 4)
    }


# ==========================================================
# Run Dataset (repeated hold-out)
# ==========================================================

def run_dataset(df, dataset_name, repeats=10, alpha=0.5, class_priors=None):

    X_df, y, s, cat_cols = preprocess_data(df, dataset_name)

    results = repeated_experiment(
        X_df, y, s, dataset_name, cat_cols,
        repeats=repeats, alpha=alpha, class_priors=class_priors
    )

    summary = summarize_results(results)

    print("\n======================")
    print(dataset_name, "(repeated hold-out)")
    print("======================")
    print(summary)

    print("\nWilcoxon Tests")

    print("\nFull BMNB vs Threshold Only (Fairness)")
    print(wilcoxon_test(results, "Full BMNB", "Threshold Only", "FS"))

    print("\nFull BMNB vs Threshold Only (Accuracy)")
    print(wilcoxon_test(results, "Full BMNB", "Threshold Only", "accuracy"))

    print("\nFull BMNB vs BLE Only (Fairness)")
    print(wilcoxon_test(results, "Full BMNB", "BLE Only", "FS"))

    print("\nFull BMNB vs BLE Only (Accuracy)")
    print(wilcoxon_test(results, "Full BMNB", "BLE Only", "accuracy"))

    return results


# ==========================================================
# Run Dataset (genuine k-fold CV)
# ==========================================================

def run_dataset_kfold(df, dataset_name, n_splits=5, alpha=0.5, class_priors=None):

    X_df, y, s, cat_cols = preprocess_data(df, dataset_name)

    methods = {"Threshold Only": "threshold", "BLE Only": "ble", "Full BMNB": "full"}
    results = {}

    for method, mode in methods.items():
        print("\nRunning (k-fold):", method)
        results[method] = evaluate_bmnb_kfold(
            X_df, y, s, dataset_name, cat_cols,
            alpha=alpha, class_priors=class_priors,
            n_splits=n_splits, mode=mode, verbose=True
        )

    summary = summarize_results(results)

    print("\n======================")
    print(dataset_name, f"({n_splits}-fold CV)")
    print("======================")
    print(summary)

    # ------------------------------------------------------------
    # Wilcoxon signed-rank tests across folds.
    # Valid as a PAIRED test here because all three methods above
    # were evaluated with the same n_splits and the same random_state,
    # so fold i's train/test partition is identical across methods -
    # each entry in results["Full BMNB"][i] is paired with the same
    # held-out fold as results["Threshold Only"][i], etc.
    # ------------------------------------------------------------
    print("\nWilcoxon Tests (paired across folds)")

    print("\nFull BMNB vs Threshold Only (Fairness)")
    print(wilcoxon_test(results, "Full BMNB", "Threshold Only", "FS"))

    print("\nFull BMNB vs Threshold Only (Accuracy)")
    print(wilcoxon_test(results, "Full BMNB", "Threshold Only", "accuracy"))

    print("\nFull BMNB vs BLE Only (Fairness)")
    print(wilcoxon_test(results, "Full BMNB", "BLE Only", "FS"))

    print("\nFull BMNB vs BLE Only (Accuracy)")
    print(wilcoxon_test(results, "Full BMNB", "BLE Only", "accuracy"))

    return results


# ==========================================================
# Main
# ==========================================================

def main():

    adult_df = pd.read_csv("adult.csv")
#     propublica_df = pd.read_csv("propublica_data_for_fairml.csv")
#     framingham_df = pd.read_csv("framingham.csv")

    adult_results = run_dataset(adult_df, "adult")
#     propublica_results = run_dataset(propublica_df, "propublica")

#     framingham_results = run_dataset(framingham_df, "framingham")

    # Uncomment for a genuine k-fold CV run (e.g. to directly address
    # a reviewer comment requesting k-fold rather than repeated hold-out):
#     framingham_kfold_results = run_dataset_kfold(framingham_df, "framingham", n_splits=5)
#     propublica_kfold_results = run_dataset_kfold(propublica_df, "propublica", n_splits=5)
    adult_kfold_results = run_dataset_kfold(adult_df, "adult", n_splits=5)




if __name__ == "__main__":
    main()


Running: Threshold Only
  Repetition 1/10
  Repetition 2/10
  Repetition 3/10
  Repetition 4/10
  Repetition 5/10
  Repetition 6/10
  Repetition 7/10
  Repetition 8/10
  Repetition 9/10
  Repetition 10/10

Running: BLE Only
  Repetition 1/10
  Repetition 2/10
  Repetition 3/10
  Repetition 4/10
  Repetition 5/10
  Repetition 6/10
  Repetition 7/10
  Repetition 8/10
  Repetition 9/10
  Repetition 10/10

Running: Full BMNB
  Repetition 1/10
  Repetition 2/10
  Repetition 3/10
  Repetition 4/10
  Repetition 5/10
  Repetition 6/10
  Repetition 7/10
  Repetition 8/10
  Repetition 9/10
  Repetition 10/10

adult (repeated hold-out)
           Method       Accuracy      Precision         Recall             F1  \
0  Threshold Only  0.794 ± 0.003  0.631 ± 0.014  0.347 ± 0.006  0.448 ± 0.006   
1        BLE Only  0.820 ± 0.002  0.634 ± 0.010  0.595 ± 0.025  0.614 ± 0.010   
2       Full BMNB  0.803 ± 0.005  0.638 ± 0.013  0.424 ± 0.024  0.509 ± 0.020   

              BI             FS          